# TRASER on the paper's test sets (Table 2: triplet, relation, object)

Kaggle: **GPU T4 ×2**, **Internet ON**. One block per test set; each block is *download → prepare → run → save*.

| block | what | when |
|---|---|---|
| **0. Setup** | repo, packages, TRASER checkpoint + Qwen processor, GPU check, one-time cleanup | every session, first |
| **1. PVSG** | 62 validation videos (VidOR, Epic-Kitchens, Ego4D parts) | its own session |
| **2. SVG2test** | 100 videos: 67 from VSPW + 33 from SA-V (paper Sec. 3) | its own session |
| **3. VidOR** | validation videos (SAM 2 turns the boxes into masks) | later (`RUN_VIDOR = False` for now) |
| **4. Score** | judge (Kimi K3) + table next to the paper's | any time, no GPU needed |

Predictions are written straight into the repo (`results/traser_bench/<dataset>/preds/`): **push at the end of each block**. Running a block again skips the videos that already have a prediction.

Settings (same as the released inference code): 1 frame/s, at most 128 frames, at most 40 objects, coverage 0.5, 4 s windows, greedy decoding. On a T4: float16 and a memory-saving attention path (same result, avoids the out-of-memory errors).

## 0. Setup (run every session)

In [ ]:
import os, sys, json, glob, time, shutil, subprocess, getpass, random
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
r = subprocess.run("python3 -m pip install -q openai huggingface_hub gdown", shell=True, capture_output=True, text=True)
print(r.stderr[-300:])
sys.path.insert(0, os.path.join(REPO, "tools"))
import bench_data as B, bench_eval as E, pvsg_data
BIG = "/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp"     # big downloads (not in the repo)
BENCH = os.path.join(BIG, "traser_bench")                             # prepared videos: video.mp4 + masks + labels
RUNS = "/kaggle/working/traser_bench_runs"                            # worker logs and progress
for d in (BENCH, RUNS):
    os.makedirs(d, exist_ok=True)
print("code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("big data →", BIG, "| free disk:", shutil.disk_usage(BIG).free // 2**30, "GB")

In [ ]:
# TRASER checkpoint (~8 GB) + the Qwen2.5-VL processor files, downloaded once (both GPU workers then read the cache)
from huggingface_hub import snapshot_download
t0 = time.time()
traser_dir = snapshot_download("UWGZQ/TRASER", ignore_patterns=["example/*"])
qwen_dir = snapshot_download("Qwen/Qwen2.5-VL-3B-Instruct", ignore_patterns=["*.safetensors", "*.bin", "*.pt", "*.pth"])
size = lambda d: sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, "**"), recursive=True) if os.path.isfile(p)) / 2**30
print(f"TRASER: {size(traser_dir):.1f} GB | Qwen processor: {size(qwen_dir) * 1024:.0f} MB | {time.time() - t0:.0f} s")

import torch
print(f"\n{torch.cuda.device_count()} GPU(s):")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cc = torch.cuda.get_device_capability(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB, compute {cc[0]}.{cc[1]} → "
          + ("bfloat16" if cc[0] >= 8 else "float16 + memory-saving attention"))

**One-time cleanup (SVG2test).** The first SVG2test run built the 67 VSPW videos from VIPSeg pictures (only every 3rd frame). Those predictions are deleted here so they are redone with the real VSPW frames. The 19 SA-V predictions that worked are kept (same settings); the 14 SA-V videos that ran out of memory are retried. Runs only once (a marker file is left behind).

In [ ]:
marker = "results/traser_bench/svg2test/.vspw_cleanup_done"
if os.path.exists("results/traser_bench/svg2test") and not os.path.exists(marker):
    gone = 0
    for p in glob.glob("results/traser_bench/svg2test/preds/*"):
        if not os.path.basename(p).startswith("sav_"):
            os.remove(p); gone += 1
    kept = len([p for p in glob.glob("results/traser_bench/svg2test/preds/sav_*.json") if not p.endswith(".stats.json")])
    open(marker, "w").write("VIPSeg-based predictions removed; VSPW frames from now on\n")
    print(f"removed {gone} files of VIPSeg-based predictions; kept {kept} SA-V predictions")
else:
    print("nothing to clean")

In [ ]:
# Helpers used by every block: start one background worker per GPU, show progress, show what to push.
def n_done(dataset):
    return len([p for p in glob.glob(f"results/traser_bench/{dataset}/preds/*.json") if not p.endswith(".stats.json")])

def start_workers(dataset, videos, extra=""):
    running = subprocess.run("pgrep -f [t]ools/bench_run.py", shell=True, capture_output=True, text=True).stdout.split()
    if running:
        print("workers already running:", running, "(wait for them, then run this cell again)")
        return
    done = {os.path.basename(p)[:-5] for p in glob.glob(f"results/traser_bench/{dataset}/preds/*.json")
            if not p.endswith(".stats.json")}
    todo = [v for v in videos if v not in done]
    free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.split()
    print(f"{dataset}: {len(done)} done earlier, {len(todo)} to run | free MiB per GPU: {free}")
    if not todo:
        return
    if any(int(f) < 12000 for f in free):
        print("!! a GPU has < 12 GB free: shut down other notebook kernels first")
    for p in glob.glob(os.path.join(RUNS, f"{dataset}_gpu*.json")):
        os.remove(p)                                          # old progress files of this dataset
    n_gpu = max(1, len(free))
    for g in range(n_gpu):
        queue = todo[g::n_gpu]
        if not queue:
            continue
        qfile = os.path.join(RUNS, f"{dataset}_gpu{g}.queue")
        open(qfile, "w").write("\n".join(queue))
        cmd = (f"CUDA_VISIBLE_DEVICES={g} HF_HOME=/tmp/hf_cache nohup python3 tools/bench_run.py --dataset {dataset} "
               f"--bench {BENCH} --runs {RUNS} --gpu_name gpu{g} {extra} --videos $(cat {qfile}) "
               f">> {RUNS}/{dataset}_gpu{g}.log 2>&1 &")
        subprocess.run(cmd, shell=True)
        print(f"  GPU {g}: {len(queue)} videos")

def bar(d, t, w=30):
    t = max(int(t or 0), 1); d = min(int(d or 0), t)
    return "█" * round(w * d / t) + "░" * (w - round(w * d / t)) + f" {d}/{t}"

def watch(dataset, total):
    from IPython.display import clear_output
    while True:
        clear_output(wait=True)
        files = sorted(glob.glob(os.path.join(RUNS, f"{dataset}_gpu*.json")))
        print(f"{time.strftime('%H:%M:%S')}   {dataset}: {n_done(dataset)} of {total} videos predicted (all sessions)\n")
        if not files:
            print("no workers were started (nothing left to run?)"); return
        states = []
        for p in files:
            st = json.load(open(p)); states.append(st["state"])
            print(f"{os.path.basename(p)[:-5]:16} {bar(st['done'], st['total'])}   {st['state']}   now: {st.get('current') or '-'}"
                  f"   failed: {len(st['failed'])}   (last update {st.get('updated', '-')})")
            log = p[:-5] + ".log"
            errs = [l for l in open(log).read().splitlines() if "FAILED" in l] if os.path.exists(log) else []
            if errs:
                print("   last error:", errs[-1][:220])
        if all(s == "finished" for s in states):
            print(f"\nall workers finished → push now (cell below), then stop the session if nothing else is left"); return
        time.sleep(30)

def push_hint(dataset):
    print(f"{n_done(dataset)} {dataset} predictions in results/traser_bench/{dataset}/preds. In a terminal:\n")
    print("cd /kaggle/working/svg2-implementation")
    print("git add results/traser_bench")
    print(f'git commit -m "TRASER bench: {dataset} predictions"')
    print("git push")

## 1. PVSG: download → prepare → run → save

**1a. Download.** `pvsg.json` + videos + masks of all three parts from Hugging Face (`Jingkang/PVSG`); only the **62 validation videos** are copied out. If it says a zip is missing, send the printed file list.

In [ ]:
t0 = time.time()
PVSG_ROOT = os.path.join(BIG, "pvsg")
paths = pvsg_data.download(PVSG_ROOT, sources=("vidor", "epic_kitchen", "ego4d"))
for name, p in paths.items():
    print(f"{name:28} {os.path.getsize(p) / 2**30:6.2f} GB")
shutil.copy(paths["pvsg.json"], os.path.join(PVSG_ROOT, "pvsg.json"))
anno = pvsg_data.load_anno(PVSG_ROOT)
PVSG_VAL = sorted(B.pvsg_val_ids(anno))
print(f"\nPVSG validation: {len(PVSG_VAL)} videos  " +
      str({s: len(anno['split'][s].get('val', [])) for s in anno['split']}) + f"  ({time.time() - t0:.0f} s)")
zips = {os.path.basename(p): p for p in paths.values()}
bad = []
for v, rep in pvsg_data.extract_videos(PVSG_ROOT, zips, anno, PVSG_VAL).items():
    chk = pvsg_data.check_video(PVSG_ROOT, anno, v)
    if not chk["ok"]:
        bad.append((v, chk))
print("frame count problems (video, check):", bad or "none")

**1b. Prepare.** Masks from the PNGs (only on the frames TRASER reads), human labels in our format, each video copied into its own folder.

In [ ]:
t0 = time.time()
for i, v in enumerate(PVSG_VAL):
    if not B.item_ready(BENCH, "pvsg", v):
        B.prepare_pvsg(PVSG_ROOT, anno, v, BENCH, E.RESULTS)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(PVSG_VAL)} prepared ({time.time() - t0:.0f} s)")
print(f"PVSG ready: {sum(B.item_ready(BENCH, 'pvsg', v) for v in PVSG_VAL)} of {len(PVSG_VAL)}")

**1c. Run** (background, one worker per GPU) and **1d. watch** (refreshes every 30 s; stopping the watch cell does not stop the workers).

In [ ]:
start_workers("pvsg", PVSG_VAL)

In [ ]:
watch("pvsg", len(PVSG_VAL))

**1e. Save** before the session ends:

In [ ]:
push_hint("pvsg")

## 2. SVG2test: download → prepare → run → save

**2a. Labels and masks** from the authors' dataset (`data/SVG2_test`, `masks/SVG2_test`). The paper builds SVG2test from **SA-V and VSPW**; the release calls the VSPW part "vipseg".

In [ ]:
t0 = time.time()
SVG2_ROOT = os.path.join(BIG, "svg2")
B.download_svg2test(SVG2_ROOT)
rows, masks = B.svg2test_rows(SVG2_ROOT), B.svg2test_masks(SVG2_ROOT)
VSPW_IDS = sorted(v for v, r in rows.items() if r["split"] == "vipseg")
SAV_IDS = sorted(v for v, r in rows.items() if r["split"] == "sav")
SVG2_ALL = sorted(rows)
sizes = sorted({tuple(masks[v][0][0]["size"]) for v in VSPW_IDS})
print(f"SVG2test: {len(VSPW_IDS)} VSPW + {len(SAV_IDS)} SA-V videos  ({time.time() - t0:.0f} s)")
print("mask sizes (height, width) of the VSPW videos:", sizes)

**2b. VSPW frames.** Downloaded from the VSPW authors' Google Drive (https://github.com/VSPW-dataset/VSPW-dataset-download). The **480p** version is used when the masks are at most 480 pixels high, otherwise the **full** version (43 GB). If Google refuses ("too many downloads"), download it yourself, add it as a Kaggle dataset and set `VSPW_ARCHIVE` to its path. Every video gets exactly one frame per mask frame, at 6 frames/s (the human relation times end at frames ÷ 6).

In [ ]:
VSPW_ARCHIVE = None          # e.g. "/kaggle/input/vspw/VSPW_480p.tar"; None = download from Google Drive
max_h = max(h for h, w in sizes)
if VSPW_ARCHIVE is None:
    if max_h <= 480:
        VSPW_ARCHIVE, drive_id = os.path.join(BIG, "VSPW_480p.archive"), "1rRujAmy3mzYqI0NjrdyVANDfrnFnBEp1"
    else:
        VSPW_ARCHIVE, drive_id = os.path.join(BIG, "VSPW_full.archive"), "14yHWsGneoa1pVdULFk7cah3t-THl7yEz"
    if not os.path.exists(VSPW_ARCHIVE):
        import gdown
        print("masks up to", max_h, "px high → downloading", os.path.basename(VSPW_ARCHIVE))
        gdown.download(id=drive_id, output=VSPW_ARCHIVE, quiet=False)
print("VSPW archive:", os.path.getsize(VSPW_ARCHIVE) // 2**30, "GB")
t0 = time.time()
found = B.extract_vspw_frames(VSPW_ARCHIVE, VSPW_IDS, os.path.join(BIG, "vspw_frames"))
print(f"VSPW frames found for {len(found)} of {len(VSPW_IDS)} videos ({time.time() - t0:.0f} s)")

SVG2_VIDEOS, SVG2_FALLBACK = {}, {}
for v in sorted(found):
    fdir = os.path.join(BIG, "vspw_frames", v)
    n = len(glob.glob(os.path.join(fdir, "*.jpg")))
    h, w = masks[v][0][0]["size"]
    out = os.path.join(BIG, "svg2test_videos", f"{v}.mp4")
    if not os.path.exists(out):
        B.frames_to_mp4(fdir, out, fps=6, size=(w - w % 2, h - h % 2))
    SVG2_VIDEOS[v] = out
    if n != len(masks[v]):              # not expected; then each mask frame gets the nearest frame (2d)
        SVG2_FALLBACK[v] = (n, len(masks[v]))
print("VSPW videos built:", len(SVG2_VIDEOS), "| frame count differs from the masks (video: frames, mask frames):",
      SVG2_FALLBACK or "none")

**2c. SA-V videos** from `sav_test.tar` with Meta's links file. Get a fresh links file (they expire), upload it to the notebook (Add input → Upload) or download it in a terminal, and set `SAV_LINKS`.

In [ ]:
SAV_LINKS = "/kaggle/working/sav_links.txt"     # Meta's links file (file_name <tab> url per line)
missing = B.fetch_sav_videos(B.sav_links(SAV_LINKS), SAV_IDS, os.path.join(BIG, "svg2test_videos"))
for v in SAV_IDS:
    p = os.path.join(BIG, "svg2test_videos", f"{v}.mp4")
    if os.path.exists(p):
        SVG2_VIDEOS[v] = p
print(f"SA-V videos: {len([v for v in SAV_IDS if v in SVG2_VIDEOS])} of {len(SAV_IDS)}; not found:", sorted(missing) or "none")

**2d. Prepare** (self-contained folders, masks kept only on the frames TRASER reads) and check every video's frame count against its masks. Expected: only SA-V videos 1 frame short (1/24 s); anything else, send the output.

In [ ]:
t0 = time.time()
for v, p in sorted(SVG2_VIDEOS.items()):
    fb = os.path.join(BIG, "vspw_frames", v) if v in SVG2_FALLBACK else None
    B.prepare_svg2test(v, rows[v], masks[v], p, BENCH, E.RESULTS, vipseg_frames=fb)
print(f"prepared {len(SVG2_VIDEOS)} of {len(SVG2_ALL)} SVG2test videos ({time.time() - t0:.0f} s)")
diff = []
for v in sorted(SVG2_VIDEOS):
    gt = B.load_gt(BENCH, "svg2test", v)
    if gt["n_frames"] != gt["mask_frames"]:
        diff.append((v, gt["n_frames"], gt["mask_frames"]))
print("video frames ≠ mask frames (video, video frames, mask frames):", diff or "none")

**2e. Run** and **2f. watch**.

In [ ]:
start_workers("svg2test", sorted(SVG2_VIDEOS))

In [ ]:
watch("svg2test", len(SVG2_ALL))

**2g. Save** before the session ends:

In [ ]:
push_hint("svg2test")

## 3. VidOR: download → prepare → run → save (later)

Validation videos and box annotations from Hugging Face (`shangxd/vidor`). Like the paper (Sec. 4.3), SAM 2 turns VidOR's boxes into masks; this happens inside the workers, on the GPU, just before each video. `N_VIDOR` = a fixed random sample (same videos every session), `None` = all 835. Set `RUN_VIDOR = True` to use this block.

In [ ]:
RUN_VIDOR = False
N_VIDOR = 200
if RUN_VIDOR:
    t0 = time.time()
    VIDOR_ROOT = os.path.join(BIG, "vidor")
    B.download_vidor(VIDOR_ROOT)
    idx = B.vidor_index({"videos": os.path.join(VIDOR_ROOT, "videos"), "anno": os.path.join(VIDOR_ROOT, "anno")})
    VIDOR_IDS = sorted(idx)
    if N_VIDOR:
        VIDOR_IDS = sorted(random.Random(0).sample(VIDOR_IDS, min(N_VIDOR, len(VIDOR_IDS))))
    for v in VIDOR_IDS:
        if not os.path.exists(os.path.join(BENCH, "vidor", v, "gt.json")):
            B.prepare_vidor_item(*idx[v], BENCH, E.RESULTS)
    print(f"VidOR: {len(VIDOR_IDS)} of {len(idx)} videos prepared ({time.time() - t0:.0f} s)")
else:
    print("VidOR skipped (RUN_VIDOR = False)")

In [ ]:
if RUN_VIDOR:
    start_workers("vidor", VIDOR_IDS)
    watch("vidor", len(VIDOR_IDS))
    push_hint("vidor")

## 4. Score (no GPU needed)

Judge: Kimi K3 on NVIDIA NIM, five categories as in the paper (identical, synonym, hypernym/hyponym, semantic overlap, mismatch); **lenient** = anything but mismatch; temporal IoU **> 0.5**. Only label pairs not judged before are sent (answers are cached in `results/traser_bench/judge_cache.json`). Scores every dataset that has predictions and writes `results/traser_bench/README.md`. Can run in a CPU-only session: run the first setup cell, then these two cells. Push afterwards.

In [ ]:
KEYS = "/root/.svg2_keys"
if not (os.path.exists(KEYS) and "NVIDIA_API_KEY=" in open(KEYS).read()):
    key = getpass.getpass("NVIDIA_API_KEY (hidden): ")
    with open(KEYS, "a") as f:
        f.write(f"NVIDIA_API_KEY={key}\n")
    os.chmod(KEYS, 0o600); del key
print("NIM key stored in", KEYS)

In [ ]:
from IPython.display import Markdown, display
report = E.evaluate(log=print)
display(Markdown(E.write_report(report)))